# 📹 GECA — Inferencia sobre Videos
Procesa videos con YOLO26 y genera Excel con métricas. Batch processing.

In [ ]:
!pip install ultralytics openpyxl -q

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Configuración

In [ ]:
import os

SHARED = '/home/jovyan/shared'
MODEL_PATH = f'{SHARED}/models/geca_Bu_Bublik_v2_v1_best.pt'

VIDEOS = [
    f'{SHARED}/videos/Bu_Bublik_v2.mp4',
    f'{SHARED}/videos/Cobolli_Tu_v2.mp4',
]

FPS_PROCESS = 10
CONF_THRESHOLD = 0.25
IMG_SIZE = 640
BATCH_SIZE = 32
OUTPUT_DIR = f'{SHARED}/results'

os.makedirs(OUTPUT_DIR, exist_ok=True)
assert os.path.exists(MODEL_PATH), f'Modelo no encontrado: {MODEL_PATH}'
for v in VIDEOS: assert os.path.exists(v), f'Video no encontrado: {v}'
print(f'Modelo: {os.path.basename(MODEL_PATH)}')
print(f'Videos: {len(VIDEOS)}, FPS: {FPS_PROCESS}, Batch: {BATCH_SIZE}')

## 2. Cargar Modelo

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL_PATH)
class_names = model.names
print(f'Clases ({len(class_names)}):')
for k, v in class_names.items(): print(f'  {k}: {v}')

## 3. Función de Inferencia (Batch)

In [ ]:
import cv2, numpy as np
from collections import defaultdict
from tqdm.notebook import tqdm

def process_video(video_path, model, fps_process=10, conf=0.25, imgsz=640, batch_size=32):
    cap = cv2.VideoCapture(video_path)
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames_video / video_fps
    frame_interval = max(1, int(video_fps / fps_process))
    total_to_process = total_frames_video // frame_interval
    video_name = os.path.splitext(os.path.basename(video_path))[0]

    print(f'\n{"="*60}')
    print(f'  {video_name}: {duration/60:.1f} min, {total_to_process} frames @ batch {batch_size}')
    print(f'{"="*60}')

    class_det = defaultdict(int)
    class_fc = defaultdict(int)
    processed = 0
    pbar = tqdm(total=total_to_process, desc=video_name, unit='f')
    idx = 0
    batch = []

    def run_batch(b):
        nonlocal processed
        if not b: return
        for r in model(b, conf=conf, imgsz=imgsz, verbose=False):
            fc = set()
            for box in r.boxes:
                cn = class_names.get(int(box.cls.item()), '?')
                class_det[cn] += 1
                fc.add(cn)
            for cn in fc: class_fc[cn] += 1
        processed += len(b)
        pbar.update(len(b))

    while True:
        ret, frame = cap.read()
        if not ret: break
        if idx % frame_interval == 0:
            batch.append(frame)
            if len(batch) >= batch_size:
                run_batch(batch)
                batch = []
        idx += 1
    run_batch(batch)
    pbar.close()
    cap.release()

    metrics = []
    for cn in sorted(class_det, key=lambda x: -class_det[x]):
        td, fw = class_det[cn], class_fc[cn]
        metrics.append({'Etiqueta': cn, 'Total detecciones': td, 'Frames con deteccion': fw,
            'Media cuando aparece': round(td/fw, 6) if fw else 0,
            'Media total': round(td/processed, 6) if processed else 0,
            'Tiempo pantalla (s)': round(fw/fps_process, 1),
            'Porcentaje tiempo (%)': round(fw/processed*100, 6) if processed else 0})

    return {'video_name': video_name, 'duration': duration, 'frames_processed': processed, 'metrics': metrics}

## 4. Procesar Videos

In [ ]:
import time
all_results = []
for vp in VIDEOS:
    t0 = time.time()
    r = process_video(vp, model, FPS_PROCESS, CONF_THRESHOLD, IMG_SIZE, BATCH_SIZE)
    elapsed = time.time() - t0
    all_results.append(r)
    print(f'  ⏱ {elapsed:.1f}s — {r["frames_processed"]/elapsed:.0f} f/s')
    print(f'  {"Etiqueta":<20} {"Det":>10} {"Frames":>10} {"Tiempo":>10} {"%":>8}')
    for m in r['metrics']:
        print(f'  {m["Etiqueta"]:<20} {m["Total detecciones"]:>10,} {m["Frames con deteccion"]:>10,} {m["Tiempo pantalla (s)"]:>10.1f} {m["Porcentaje tiempo (%)"]:>8.2f}')
print(f'\n✓ {len(all_results)} videos')

## 5. Exportar Excel

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

def export_excel(result, output_dir):
    wb = Workbook()
    ws = wb.active
    ws.title = 'Métricas'
    hf = Font(bold=True, color='FFFFFF', size=11)
    hfill = PatternFill(start_color='2E3440', end_color='2E3440', fill_type='solid')
    b = Border(left=Side(style='thin',color='CCCCCC'), right=Side(style='thin',color='CCCCCC'),
              top=Side(style='thin',color='CCCCCC'), bottom=Side(style='thin',color='CCCCCC'))
    headers = ['Etiqueta','Total detecciones','Frames con deteccion','Media cuando aparece','Media total','Tiempo pantalla (s)','Porcentaje tiempo (%)']
    for col, h in enumerate(headers, 1):
        c = ws.cell(row=1, column=col, value=h)
        c.font, c.fill, c.alignment, c.border = hf, hfill, Alignment(horizontal='center'), b
    for ri, m in enumerate(result['metrics'], 2):
        for col, key in enumerate(headers, 1):
            c = ws.cell(row=ri, column=col, value=m[key])
            c.border = b
    ws.column_dimensions['A'].width = 20
    for col in 'BCDEFG': ws.column_dimensions[col].width = 22
    fp = os.path.join(output_dir, f'{result["video_name"]}_metrics.xlsx')
    wb.save(fp)
    return fp

for r in all_results:
    print(f'✓ {export_excel(r, OUTPUT_DIR)}')
print(f'\n📂 Windows: \\\\10.43.13.186\\Compartida\\results\\')

## 6. MLflow (Opcional)

In [ ]:
import mlflow
mlflow.set_tracking_uri('http://geca_mlflow:5000')
mlflow.set_experiment('GECA_Inference')
for r in all_results:
    with mlflow.start_run(run_name=f'inference_{r["video_name"]}'):
        mlflow.log_param('video', r['video_name'])
        mlflow.log_param('model', os.path.basename(MODEL_PATH))
        mlflow.log_param('batch_size', BATCH_SIZE)
        for m in r['metrics']:
            mlflow.log_metric(f'{m["Etiqueta"]}_pct', m['Porcentaje tiempo (%)'])
        ep = os.path.join(OUTPUT_DIR, f'{r["video_name"]}_metrics.xlsx')
        if os.path.exists(ep): mlflow.log_artifact(ep)
        print(f'✓ {r["video_name"]}')
print(f'Ver: http://10.43.13.186:5000')